# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a practical, step-by-step workflow for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library. All entities in the dataset (record sets, fields, columns, etc.) are referenced by their unique `@id` fields for reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure the mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

This section loads the dataset metadata and prepares the dataset for exploration using `mlcroissant`. 

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Published:", getattr(metadata, 'datePublished', 'N/A'))

## 2. Data Overview

In this section, we list all record sets in the dataset (datasets, tables, etc.), including their `@id` fields and field structure. All references use their `@id` to ensure unambiguous access. 

In [ ]:
# List all record sets in the dataset
# Each record set has a unique '@id'

record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in dataset metadata.")
else:
    print(f'Found {len(record_sets)} record set(s):\n')
    for rs in record_sets:
        print(f"Record set: {rs.name}")
        print(f"  @id: {rs['@id']}")
        # List field @id's
        try:
            field_ids = [field['@id'] for field in rs.fields]
            print(f"  Fields: {field_ids}")
        except Exception as e:
            print(f"  Could not list fields: {e}")
        print("")
    # Print columns in the first record set for preview
    first_rs = record_sets[0]
    print(f"First record set columns: {[col['@id'] for col in first_rs.columns]}")

## 3. Data Extraction

Let's load all records for each record set. We'll use the record set `@id` and column `@id`s identified above. Each DataFrame is stored in a dictionary using the record set `@id` as the key.

In [ ]:
# If record_sets is empty, try to get their @ids and exit early
if not record_sets:
    print("No record sets available for extraction.")
else:
    # List all record set @ids
    record_set_ids = [rs['@id'] for rs in record_sets]
    print(f"Record set @ids: {record_set_ids}")

    # Load the data for each record set using its @id
    dataframes = {}
    for rs in record_sets:
        rs_id = rs['@id']
        print(f'Loading records from record set {rs_id} ...')
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        print(f"  Shape: {df.shape}")
        dataframes[rs_id] = df
        # Print columns
        print(f'  Columns: {df.columns.tolist()}')
        print()

    # For notebook demonstration, pick first available record set and its first numeric column
    main_record_set_id = record_set_ids[0]
    print(f"Main record set for further steps: {main_record_set_id}")
    print(f"Preview: \n{dataframes[main_record_set_id].head()}")

## 4. Exploratory Data Analysis (EDA)

We demonstrate common data processing steps: filtering records by value, normalizing a numeric field, and grouping by another field. All field references use their `@id` as column names.

Replace the `numeric_field_id` and `group_field_id` below with a field available in the main record set, as listed above.

In [ ]:
# Example EDA -- fill in with available fields

main_df = dataframes[main_record_set_id]
# For demonstration, try to pick a numeric column
numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Numeric field: {numeric_field_id}")
else:
    print("No numeric fields found; using first column for demonstration.")
    numeric_field_id = main_df.columns[0]

# Filtering: select values above the mean
threshold = main_df[numeric_field_id].mean()
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field
categorical_fields = [col for col in main_df.columns if main_df[col].dtype == 'object']
if categorical_fields:
    group_field_id = categorical_fields[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped mean values by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No categorical fields to group by.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and the relationship with a categorical field, if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
main_df[numeric_field_id].hist(ax=ax[0], bins=30, color='skyblue')
ax[0].set_title(f"Distribution of {numeric_field_id}")
ax[0].set_xlabel(numeric_field_id)
ax[0].set_ylabel("Count")

# Boxplot grouped by a categorical field (if any)
if categorical_fields:
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id, ax=ax[1])
    ax[1].set_title(f'{numeric_field_id} by {group_field_id}')
    plt.setp(ax[1].xaxis.get_majorticklabels(), rotation=45)
else:
    ax[1].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Conclusion

- We successfully loaded Croissant metadata and records from the FAIR<sup>2</sup> dataset using `mlcroissant`.
- By referencing all entities via their `@id`, data extraction and processing are reproducible.
- The exploratory analysis demonstrated common EDA steps, such as filtering and normalizing by a numeric field and grouping by a category.
- Use this notebook as a template for further exploration and analysis of Croissant-formatted datasets.